# MNIST Digit Image Generation (Simple Autoencoder-style demo)

This fixes the shape mismatch in the supplied code. It trains a small autoencoder on MNIST digit 9 images and saves reconstructions. **This is not a true generative VAE**; it demonstrates image reconstruction rather than reliable new-image generation.

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# CPU works; CUDA is used automatically when available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.ToTensor()
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
indices = [i for i, (_, label) in enumerate(dataset) if label == 9][:1000]
loader = DataLoader(Subset(dataset, indices), batch_size=64, shuffle=True)
print("Device:", device, "| training examples:", len(indices))

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 32), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(32, 128), nn.ReLU(), nn.Linear(128, 784), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z).view(-1, 1, 28, 28)

model = Autoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()

# Keep epochs low for a quick notebook demo; increase to 5-10 for better reconstructions.
for epoch in range(3):
    model.train()
    total_loss = 0.0
    for images, _ in loader:
        images = images.to(device)
        reconstructed = model(images)
        loss = loss_fn(reconstructed, images)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    print(f"Epoch {epoch + 1}/3 - loss: {total_loss / len(indices):.4f}")

In [ ]:
model.eval()
images, _ = next(iter(loader))
with torch.no_grad():
    reconstructed = model(images.to(device)).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(images[i, 0], cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(reconstructed[i, 0], cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Reconstructed")
plt.tight_layout(); plt.show()

os.makedirs("generated_digits", exist_ok=True)
for i, image in enumerate(reconstructed[:9]):
    plt.imsave(f"generated_digits/digit9_{i}.png", image[0].numpy(), cmap="gray", vmin=0, vmax=1)
print("Saved 9 reconstructed digit images in generated_digits/")